# Microsoft Sentinel Identity Investigation Playground (Notebook's 101)
This notebook searches the last years worth of data in the Sentinel data lake for a user principal name (UPN) across:
- `SigninLogs`
- `AuditLogs`

Note: You must be ingesting these tables either direcly into Microsoft Sentinel data lake, or into the Analytic tier (in whichs case, they will be mirrored in the lake).

This notebook contains basic queries and visualitaions powered by data in Microsoft Sentinel data lake. The purpose being to act as an introducion to the 'art of the possible' that Notebooks can provide.

This notebook is planned to have updates made regularly, so be sure to check the GitHub for any updated version.

Note: This Notebook is an individual contribution, no an official Microsoft announcement.

In [ ]:
# Import required libraries for analysis
from sentinel_lake.providers import MicrosoftSentinelProvider # imports the MicrosoftSentinelProvider class from the sentinel_lake.providers module, which allows us to connect to and query data from Microsoft Sentinel
from pyspark.sql.functions import * # imports all functions from PySpark SQL, including col(), count(), get_json_object(), etc. for data manipulation
from pyspark.sql.types import * # imports all data types from PySpark SQL, such as StringType, IntegerType, etc. for defining schemas
from pyspark.sql import Window # imports the Window class for performing windowed operations like running totals or rankings
import matplotlib.pyplot as plt  # imports matplotlib for creating charts and visualizations
from datetime import datetime, timedelta, timezone # imports datetime for working with dates/times, timedelta for calculating time differences, and timezone for UTC-aware timestamps

# Initialize the data provider
data_provider = MicrosoftSentinelProvider(spark) # creates an instance of the MicrosoftSentinelProvider class using the existing Spark session, enabling us to read/write Sentinel data

In [ ]:
# === ANALYSIS CONFIGURATION ===
# Adjust these parameters to fine-tune the anomaly detection

# Time window configurations (in days)
LOOKBACK_DAYS = 15      # defines how many days of historical data to analyze when establishing baseline user behavior patterns
User_UPN = "<user@domain.com>" # specifies the user principal name (UPN) to investigate - replace with the target user's UPN, e.g. "user@contoso.com"
WORKSPACE_NAME = "<your-sentinel-workspace-name>" # specifies the Microsoft Sentinel workspace name to query data from

## Sign-in Location Summary
Query the `SigninLogs` table to identify the user's most frequent sign-in source IP addresses and associated locations over the selected lookback period. This introduces `.groupBy()` and `.agg()` for summarizing activity, along with `get_json_object()` for extracting nested location fields from JSON data.

In [ ]:
# Query SigninLogs for user authentication count by source IP and location (last 24 hours)
sign_in_logs = data_provider.read_table("SigninLogs", WORKSPACE_NAME) # reads the SigninLogs table from the specified Sentinel workspace into a Spark DataFrame. KQL equivalent: SigninLogs

sign_in_by_ip_location = ( # begins building a query to analyze sign-ins by IP and location, storing the result in sign_in_by_ip_location
    sign_in_logs # starts with the sign_in_logs DataFrame loaded above
    .filter(col("TimeGenerated") >= (datetime.now(tz=timezone.utc) - timedelta(days=15))) # KQL: | where TimeGenerated >= ago(15d) — filters to only include logs from the last 15 days
    .filter(col("UserPrincipalName") == User_UPN) # KQL: | where UserPrincipalName == "user@domain.com" — filters to only include sign-ins for the specific user
    .groupBy( # KQL: | summarize ... by — groups the filtered data by the following columns to aggregate authentication counts
        col("IPAddress"), # groups by the source IP address where the sign-in originated from
        get_json_object(col("LocationDetails"), "$.city").alias("City"), # KQL: tostring(LocationDetails.city) — extracts the city from the LocationDetails JSON field
        get_json_object(col("LocationDetails"), "$.state").alias("State"), # KQL: tostring(LocationDetails.state) — extracts the state from the LocationDetails JSON field
        get_json_object(col("LocationDetails"), "$.countryOrRegion").alias("Country") # KQL: tostring(LocationDetails.countryOrRegion) — extracts the country from the LocationDetails JSON field
    )
    .agg(count("*").alias("AuthenticationCount")) # KQL: summarize AuthenticationCount = count() — counts the number of sign-ins for each unique IP/location combination
    .orderBy(col("AuthenticationCount").desc()) # KQL: | sort by AuthenticationCount desc — sorts the results in descending order by authentication count
    .limit(5) # KQL: | top 5 by AuthenticationCount — limits the output to only the top 5 results
)

print(f"Top 5 sign-in locations for {User_UPN} in the last year:\n") # prints a header message indicating what data is being displayed
sign_in_by_ip_location.show(truncate=False) # displays the DataFrame results in the console, with truncate=False to show full column values without cutting them off

## Sign-in Success vs. Failure Analysis
Analyze the ratio of successful to failed sign-ins for the user. This uses PySpark's `when()`/`otherwise()` functions — the equivalent of KQL's `iff()` or `case` operators.

In [ ]:
# Categorize sign-ins as Success or Failure and count by status
sign_in_status = (
    sign_in_logs # starts with the sign_in_logs DataFrame loaded in the previous query cell
    .filter(col("TimeGenerated") >= (datetime.now(tz=timezone.utc) - timedelta(days=LOOKBACK_DAYS))) # KQL: | where TimeGenerated >= ago(15d) — filters to the configured lookback window
    .filter(col("UserPrincipalName") == User_UPN) # KQL: | where UserPrincipalName == "user@domain.com" — filters to the target user
    .withColumn("Status",  # KQL: | extend Status = iff(ResultType == "0", "Success", "Failure") — creates a new column using conditional logic
        when(col("ResultType") == "0", "Success")  # ResultType 0 = successful sign-in
        .otherwise("Failure")  # anything else is treated as a failure
    )
    .groupBy("Status") # KQL: | summarize ... by Status — groups rows by the Status column
    .agg(count("*").alias("Count")) # KQL: summarize Count = count() — counts the number of sign-ins per status
    .orderBy(col("Count").desc()) # KQL: | sort by Count desc — sorts results by count descending
)

print(f"Sign-in status breakdown for {User_UPN} (last {LOOKBACK_DAYS} days):\n")
sign_in_status.show(truncate=False)

## Audit Log Activity (Visualization)
Query the `AuditLogs` table to surface recent directory changes targeting the user, such as password resets, group membership changes, or role assignments. Results are visualized as a horizontal bar chart using `matplotlib`. This introduces `.select()` for choosing specific columns — similar to KQL's `project` operator, and `.toPandas()` for converting a PySpark DataFrame to a Pandas DataFrame for visualization.

In [ ]:
# Read the AuditLogs table from the Sentinel workspace. KQL equivalent: AuditLogs
audit_logs = data_provider.read_table("AuditLogs", WORKSPACE_NAME)

# Query audit events targeting the user, grouped by operation type
user_audit_summary = (
    audit_logs  # starts with the audit_logs DataFrame
    .filter(col("TimeGenerated") >= (datetime.now(tz=timezone.utc) - timedelta(days=LOOKBACK_DAYS)))  # KQL: | where TimeGenerated >= ago(15d) — filters to the lookback window
    .filter(col("TargetResources").contains(User_UPN))  # KQL: | where TargetResources has User_UPN — filters where the user is the target of the action
    .select(  # KQL: | project — selects only the columns we need
        col("OperationName"),  # the name of the audit operation (e.g., "Reset password", "Add member to group")
        col("Result")  # whether the operation succeeded or failed
    )
    .groupBy("OperationName")  # KQL: | summarize ... by OperationName — groups by the type of operation
    .agg(count("*").alias("EventCount"))  # KQL: summarize EventCount = count() — counts occurrences of each operation
    .orderBy(col("EventCount").desc())  # KQL: | sort by EventCount desc — sorts by most frequent operations
    .limit(10)  # KQL: | top 10 by EventCount — limits to top 10 operation types
)

# Convert PySpark DataFrame to Pandas for visualization
audit_pd = user_audit_summary.toPandas()  # .toPandas() converts the distributed Spark DataFrame into a local Pandas DataFrame — no direct KQL equivalent

# Create a horizontal bar chart. KQL equivalent: | render barchart
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(audit_pd["OperationName"], audit_pd["EventCount"], color="#0078D4")
ax.set_xlabel("Event Count")
ax.set_ylabel("Operation")
ax.set_title(f"Top Audit Log Operations Targeting {User_UPN} (Last {LOOKBACK_DAYS} Days)")
ax.invert_yaxis()  # puts the highest count at the top
plt.tight_layout()
plt.show()

In [ ]:
# Writes query results into a table
data_provider.save_as_table(sign_in_by_ip_location, "UPN_Investigation_SPRK_CL", WORKSPACE_NAME) # Creates a new table in the analytic tier with the _SPRK_CL suffix
print(f"\nData has been written to table in the analytic tier with the _SPRK_CL suffix\n") # prints to the console that the data has been written to the analytic tier with the _SPRK_CL suffix
print(f"Data can also be written to the lake tier with the _SPRK suffix\n") # also prints to the console that the data can also be written to the lake tier with the _SPRK suffix